### Query Translation - Multi Query
Query translation sits at the first stage of an advanced RAG pipeline. The goal of query translation is to take the input user question and to translate it in some
way as to improve retrival.

Semantic search on embeddings is hard to get right. Embedding long documents is
especially challenging. User queries are a challenge, if user provides an ambigious
query, they'll get an ambiguous matches and hence an ambguous answer (because we are doing semantic similarity searches) ! LLMs just follow what was in the context and hallucinate answers as a result.

One approach is to take the query & re-write it (or reframing it) from a different perspective. Multi-query is one such approach, another is RAG Fusion. In multi query, we break a larger user query into multiple queries; for each query we find the matching contexts, which we can combine later into a larger context for LLM to answer from. The intuition is that this improves the search results.

![Multi Query](images/multi_query.png)

In [17]:
import bs4, os
import pathlib
from dotenv import load_dotenv
from typing import List, TypedDict
from rich.console import Console
from rich.markdown import Markdown

from langchain.chat_models import init_chat_model
from langchain_core.documents import Document
from langchain_community.document_loaders import WebBaseLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter

# since we are using Gemini, we'll use Google embeddings
from langchain_google_genai import GoogleGenerativeAIEmbeddings
from langchain_community.vectorstores import FAISS

from langgraph.graph import StateGraph, START, END

In [18]:
# load API keys from .env files
load_dotenv(override=True)
# for colorful text output
console = Console()

In [19]:
# create our LLM - we'll be using Gemini-2.5-flash
llm = init_chat_model("google_genai:gemini-2.5-flash", temperature=0.0)
faiss_store = pathlib.Path(os.getcwd()) / "faiss_index_rag_mq"

In [20]:
pathlib.Path(os.getcwd())

WindowsPath('c:/Users/BHOBEMRMANISHJAGDISH/Dev/code/git_projects/learning_langchain/src/langchain_tutorial')

In [21]:
def create_or_load_embeddings():
    """creates if not available or loads from disk a FAISS embedding"""
    if not faiss_store.exists():
        # in this example we'll load document from a URL
        web_url = "https://lilianweng.github.io/posts/2023-06-23-agent/"
        console.print(
            f"[yellow]Loading document from URL {web_url}. Please wait...[/yellow]"
        )
        loader = WebBaseLoader(
            web_paths=(web_url,),
            bs_kwargs=dict(
                parse_only=bs4.SoupStrainer(
                    class_=("post-content", "post-title", "post-header")
                )
            ),
        )
        blog_docs = loader.load()

        console.print(f"[blue]Loaded {len(blog_docs)} documents from URL[/blue]")
        console.print(
            f"[blue]Metadata of first document: {blog_docs[0].metadata}[/blue]"
        )
        console.print(
            f"[blue]First 200 chars of first document: {blog_docs[0].page_content[:200]}[/blue]"
        )

        # split document into chunks of 1000 chars with 200 chars overlap
        console.print(f"[yellow]Chunking the PDF. Please wait...[/yellow]")

        text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
            chunk_size=300, chunk_overlap=50
        )

        # Make splits
        splits = text_splitter.split_documents(blog_docs)
        console.print(f"[blue]Created {len(splits)} chunks[/blue]")

        # save to embeddings
        console.print("[yellow]Creating embeddings. Please wait...[/yellow]")
        # Use a Gemini embedding model that is suitable for retrieval.
        # It is important to match the model to the task.
        embeddings = GoogleGenerativeAIEmbeddings(
            model="models/text-embedding-004",
            task_type="retrieval_document",
        )
        vector_store = FAISS.from_documents(documents=splits, embedding=embeddings)
        retriever = vector_store.as_retriever()
        vector_store.save_local(str(faiss_store))
        console.print(
            f"[yellow]Local embeddings created at {str(faiss_store)}[/yellow]"
        )
    else:
        console.print(
            f"[yellow]Loading existing embeddings from {str(faiss_store)}[/yellow]"
        )
        embeddings = GoogleGenerativeAIEmbeddings(
            model="models/text-embedding-004",
            task_type="retrieval_document",
        )
        vector_store = FAISS.load_local(
            str(faiss_store), embeddings, allow_dangerous_deserialization=True
        )
        retriever = vector_store.as_retriever()

    return retriever

In [22]:
retriever = create_or_load_embeddings()

Loading existing embeddings from 
c:\Users\BHOBEMRMANISHJAGDISH\Dev\code\git_projects\learning_langchain\src\langchain_tutorial\faiss_index_rag_mq

In [23]:
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import StrOutputParser

# from langchain_google_genai import ChatGoogleGenerativeAI

# Multi Query: Different Perspectives
template = """You are an AI language model assistant. Your task is to generate five 
different versions of the given user question to retrieve relevant documents from a vector 
database. By generating multiple perspectives on the user question, your goal is to help
the user overcome some of the limitations of the distance-based similarity search. 
Provide these alternative questions separated by newlines. 

Original question: {question}"""

prompt_perspectives = ChatPromptTemplate.from_template(template)

generate_queries = (
    prompt_perspectives
    # | ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)
    | llm
    | StrOutputParser()
    | (lambda x: x.split("\n"))
)

In [24]:
generate_queries = (
    prompt_perspectives | llm | StrOutputParser() | (lambda x: x.split("\n"))
)

# let's try invoking the chain
generate_queries.invoke({"question": "What is task decomposition for LLM agents?"})

['Define task breakdown strategies for large language model agents.',
 'How do LLM agents break down complex problems into smaller steps?',
 'Why is breaking down tasks important for AI agents powered by large language models?',
 'Explain methods for hierarchical planning or sub-goal generation in LLM-based agents.',
 'Describe the concept of modularizing tasks for autonomous language model systems.']

So you notice that we generated 5 different versions of the same query to improve our matches against the vector database.

In [ ]:
from langchain.load import dumps, loads


def get_unique_union(documents: list[list]):
    """Unique union of retrieved docs"""
    # Flatten list of lists, and convert each Document to string
    flattened_docs = [dumps(doc) for sublist in documents for doc in sublist]
    # Get unique documents
    unique_docs = list(set(flattened_docs))
    # Return
    return [loads(doc) for doc in unique_docs]

In [26]:
# Retrieve
question = "What is task decomposition for LLM agents?"
# here we are firing multiple queries against the vector store, getting all the
# responses & creating a unique set from all the responses.
retrieval_chain = generate_queries | retriever.map() | get_unique_union
docs = retrieval_chain.invoke({"question": question})
print(f"Got {len(docs)} documents")
for i, doc in enumerate(docs):
    console.print(
        Markdown(f"### Document {i+1}\n{doc.page_content[:50] + "..."}\n---\n")
    )

# and print the retrival chain too
console.print(f"Retrieval chain: {retrieval_chain}")

Got 7 documents


C:\Users\BHOBEMRMANISHJAGDISH\AppData\Local\Temp\ipykernel_19172\1875499216.py:12: LangChainBetaWarning: The function `loads` is in beta. It is actively being worked on, so the API may change.
  return [loads(doc) for doc in unique_docs]


Document 1                                                     


                                Or @article{weng2023agent, title   = "LLM-powere...

Document 2                                                     


                               The generative agent architecture. (Image source: ...

Document 3                                                     


                               (2) Model selection: LLM distributes the tasks to ...

Document 4                                                     


                               Another quite distinct approach, LLM+P (Liu et al....

Document 5                                                     


                               [4] Liu et al. “LLM+P: Empowering Large Language M...

Document 6                                                     


                               } ] Challenges# After going through key ideas and ...

Document 7                                                     


                               Component One: Planning# A complicated task usuall...

Retrieval chain: first=ChatPromptTemplate(input_variables=['question'], input_types={}, partial_variables={}, 
messages=[HumanMessagePromptTemplate(prompt=PromptTemplate(input_variables=['question'], input_types={}, 
partial_variables={}, template='You are an AI language model assistant. Your task is to generate five \ndifferent 
versions of the given user question to retrieve relevant documents from a vector \ndatabase. By generating multiple
perspectives on the user question, your goal is to help\nthe user overcome some of the limitations of the 
distance-based similarity search. \nProvide these alternative questions separated by newlines. \n\nOriginal 
question: {question}'), additional_kwargs={})]) middle=[ChatGoogleGenerativeAI(model='models/gemini-2.5-flash', 
google_api_key=SecretStr('**********'), temperature=0.0, 
client=<google.ai.generativelanguage_v1beta.services.generative_service.client.GenerativeServiceClient object at 
0x0000019E48112150>, default_metadata=()), StrOutputParser(), RunnableLambda(lambda x: x.split('\n')), 
RunnableEach(bound=VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], 
vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x0000019E48113EC0>, search_kwargs={}))] 
last=RunnableLambda(get_unique_union)

In [28]:
from operator import itemgetter

# RAG
template = """Answer the following question based on this context:

{context}

Question: {question}
"""

prompt = ChatPromptTemplate.from_template(template)

# llm = ChatGoogleGenerativeAI(model="gemini-2.5-flash", temperature=0)

final_rag_chain = (
    {"context": retrieval_chain, "question": itemgetter("question")}
    | prompt
    | llm
    | StrOutputParser()
)

response = final_rag_chain.invoke({"question": question})
console.print(Markdown(response))

Task decomposition for LLM agents is the process of breaking down a complicated task into multiple smaller,        
simpler, and more manageable steps or subgoals. This allows the agent to plan ahead and tackle complex problems    
effectively.                                                                                                       

It can be achieved through various methods:                                                                        

 1 LLM with simple prompting: Using prompts like "Steps for XYZ.\n1." or "What are the subgoals for achieving XYZ?"
 2 Task-specific instructions: Providing specific instructions relevant to the task, such as "Write a story        
   outline" for writing a novel.                                                                                   
 3 Human inputs: Direct human guidance in breaking down the task.                                                  

Techniques like Chain of Thought (CoT) and Tree of Thoughts are used to facilitate this decomposition, where CoT   
instructs the model to "think step by step," and Tree of Thoughts extends this by exploring multiple reasoning     
possibilities at each step, creating a tree structure of thoughts.